# Import libraries

In [21]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.tools import tool

import ollama
# web_tool.py

import json
import socket
import ipaddress
from urllib.parse import urlparse
from pydantic import BaseModel, Field

import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import subprocess
from datetime import datetime



# Tools

## Web search

In [95]:

def is_safe_url(url: str) -> bool:
    """Basic protection against private/internal network URLs."""

    try:
        parsed = urlparse(url)

        if parsed.scheme not in ("http", "https"):
            return False

        hostname = parsed.hostname

        if not hostname:
            return False

        if hostname.lower() in ("localhost",):
            return False

        addresses = socket.getaddrinfo(
            hostname,
            None,
            type=socket.SOCK_STREAM
        )

        for address in addresses:
            ip = ipaddress.ip_address(address[4][0])

            if not ip.is_global:
                return False

        return True

    except Exception:
        return False


def read_webpage(url: str, max_chars: int = 1500) -> str:

    if not is_safe_url(url):
        return "Blocked unsafe URL."

    try:
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 Chrome/120 Safari/537.36"
            )
        }

        response = requests.get(
            url,
            headers=headers,
            timeout=10,
            allow_redirects=True
        )

        # Validate the final URL after redirects as well.
        if not is_safe_url(response.url):
            return "Blocked unsafe redirect."

        response.raise_for_status()

        content_type = response.headers.get(
            "Content-Type", ""
        ).lower()

        if "html" not in content_type:
            return "This page is not HTML content."

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove elements that usually contain irrelevant content.
        for tag in soup([
            "script",
            "style",
            "nav",
            "footer",
            "header",
            "aside",
            "noscript",
            "svg",
            "form"
        ]):
            tag.decompose()

        title = (
            soup.title.get_text(" ", strip=True)
            if soup.title else "Untitled"
        )

        text = soup.get_text(
            separator=" ",
            strip=True
        )

        text = " ".join(text.split())

        return (
            f"Page title: {title}\n"
            f"URL: {response.url}\n"
            f"Content:\n{text[:max_chars]}"
        )

    except Exception as e:
        return f"Failed to read webpage: {str(e)}"

class WebSearchSchema(BaseModel):
    
    query:str = Field(
        description="The search query to look up on the web."
    )

@tool(args_schema=WebSearchSchema)
def web_search_and_read(query: str,num_of_search_pages: int =3) -> str:
    """
    Search the web for the user query, read relevant pages,
    and return the extracted content.
    """

    try:
        search_results = DDGS().text(
            query,
            max_results=num_of_search_pages
        )

        search_results = list(search_results)

    except Exception as e:
        return f"Search failed: {str(e)}"

    if not search_results:
        return "No search results found."

    output = []

    # Read the first 3 search results.
    for result in search_results[:num_of_search_pages]:

        title = result.get("title", "Untitled")
        url = result.get("href", "")
        snippet = result.get("body", "")

        if not url:
            continue

        page_content = read_webpage(url)

        output.append({
            "title": title,
            "url": url,
            "search_snippet": snippet,
            "page_content": page_content
        })

    return json.dumps(
        output,
        indent=2,
        ensure_ascii=False
    )

### Test Tool

In [5]:
search_query = "Python programming language"
search_results = web_search_and_read(search_query)

In [6]:
print(search_results)

[
  {
    "title": "Python (programming language)",
    "url": "https://en.wikipedia.org/wiki/Python_(programming_language)",
    "search_snippet": "Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive (\"batteries-included\") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. As of 2026, the Python Software Foundation supports Python 3.15.2, 3.14.10, 3.13.18, 3.12.16 and 3.11.18 that all were \"expedited security 

In [7]:
search_query = "how to build MCP servers"

search_results = web_search_and_read(query=search_query,num_of_search_pages=5)

In [8]:
print(search_results)

[
  {
    "title": "mcp.science/docs/how-to-build-your-own-mcp-server-step-by ...",
    "url": "https://github.com/pathintegral-institute/mcp.science/blob/main/docs/how-to-build-your-own-mcp-server-step-by-step.md",
    "search_snippet": "How to Build Your Own MCP Server — Step by Step (Beginner Friendly) Welcome! This guide is designed for researchers, scientists, and anyone with basic Python knowledge—even if you have no formal programming background.",
    "page_content": "Failed to read webpage: 429 Client Error: Too Many Requests for url: https://github.com/pathintegral-institute/mcp.science/blob/main/docs/how-to-build-your-own-mcp-server-step-by-step.md"
  },
  {
    "title": "Build an MCP server - Model Context Protocol",
    "url": "https://modelcontextprotocol.io/docs/2026-07-28/develop/build-server",
    "search_snippet": "Jul 28, 2026 · We’ll build a server that exposes two tools: get_alerts and get_forecast. Then we’ll connect the server to an MCP host (in this case, Claude

In [11]:
search_query = "what is an mcp server, how it works and how to build MCP servers"

search_results = web_search_and_read(query=search_query,num_of_search_pages=5)

In [12]:
print(search_results)

[
  {
    "title": "What Is an MCP Server? How It Works, Examples & Setup",
    "url": "https://mcpservers.org/learn/what-is-an-mcp-server",
    "search_snippet": "Jul 19, 2026 · What is an MCP server? Learn how MCP connects AI clients to tools and data, how local and remote servers work, setup examples, and security checks.",
    "page_content": "Page title: What Is an MCP Server? How It Works, Examples & Setup\nURL: https://mcpservers.org/learn/what-is-an-mcp-server\nContent:\nWhat Is an MCP Server? How It Works, Examples & Setup The short answer An MCP server gives an AI application a standard way to use an outside capability. An MCP server is a program that exposes tools, data, or reusable workflows to an AI application through the Model Context Protocol. Instead of building a different integration for every AI product, a service can implement MCP once and be discovered by compatible clients. The server does not have to be a large cloud service. It can be a local command that reads

In [15]:
search_query = "what are the latest developments in AI research?"

search_results = web_search_and_read(query=search_query,num_of_search_pages=5)

In [16]:
print(search_results)

[
  {
    "title": "Latest AI developments 2026: Models, agentic AI, infrastructure",
    "url": "https://www.netguru.com/blog/latest-ai-developments-2026",
    "search_snippet": "Aug 20, 2026 · Latest AI developments 2026: frontier model releases, agentic AI in production, infrastructure shifts, and security risks. See what actually changes your roadmap.",
    "page_content": "Page title: Latest AI developments 2026: Models, agentic AI, infrastructure\nURL: https://www.netguru.com/blog/latest-ai-developments-2026\nContent:\nLatest AI developments 2026: Models, agentic AI, infrastructure Contents Most 'AI news' roundups confuse volume with signal, a dozen model releases a week, but only one or two that change a build-versus-buy decision. For engineering leaders, the real question isn't what launched, it's what shifts your inference cost, your vendor lock-in risk, or your agentic deployment timeline. This guide filters recent frontier model releases, agentic AI production rollouts, infr

# Agent

In [96]:
def get_ollama_model(model_name,num_ctx=16046,num_predict=4046,keep_alive="2m",stream=False):
    return  ChatOllama(
            model=model_name,
            validate_model_on_init=True,
            num_ctx= num_ctx,
            num_predict=num_predict,
            stream=stream,
            keep_alive=keep_alive
        )

def stop_ollama(model:str):
    """
    Stop the Ollama model if it is running.
    """
    try:
        # Check if the model is running
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
        if model in result.stdout:
            # Stop the model
            subprocess.run(["ollama", "stop", model])
            print(f"Stopped Ollama model: {model}")
        else:
            print(f"Ollama model {model} is not running.")
    except Exception as e:
        print(f"Error stopping Ollama model: {e}")


In [97]:
date_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Current date and time: {date_time}")
SYSTEM_PROMPT = F"""
                You are a helpful assistant that can search the web for information, read relevant web pages, and provide concise summaries.
                Today is {date_time}. Use the current date and time in your responses when relevant.
                You have no knowledge after 2024. For any current or recent information you MUST call web_search_and_read.
"""

Current date and time: 2026-11-27 22:46:11


In [98]:
tools = [web_search_and_read]

In [100]:
llm = get_ollama_model(model_name="qwen3:8b",num_ctx=16046,num_predict=8046,keep_alive="2m",stream=False)
agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

In [76]:
# agent_response = agent.run("What are the latest developments in AI research?")
# print(agent_response)

query = "What are the latest developments in AI research?"
messages = []
messages.append({
                "role": "user",
                "content": query
            })

response = agent.invoke({"messages": messages})

In [77]:
# response_content = response.get("content", "")

In [78]:
print(response['messages'][-1].content)

As of 2026, key developments in AI research include:

1. **Efficiency Over Scaling**: Researchers are prioritizing smaller, more efficient models that retain high performance with reduced computational costs. Institutions like DeepMind, Meta FAIR, and university labs have developed architectures achieving near-frontier performance at a tenth of the compute cost, addressing practical deployment challenges.

2. **Agentic AI in Production**: Agentic AI (systems that autonomously plan and execute tasks) has transitioned from demos to real-world workflows, impacting industries by enabling dynamic decision-making and automation. However, this shift introduces new risks, such as vendor lock-in through model routing across platforms.

3. **Frontier Models and Benchmarks**: OpenAI and Google’s large language models (LLMs) are now benchmarking closely, narrowing the gap in capabilities. This competition drives innovation but also raises concerns about infrastructure costs and security vulnerabil

In [79]:
from IPython.display import display, Markdown
display(Markdown(f"**Agent Response:** {response['messages'][-1].content}"))

**Agent Response:** As of 2026, key developments in AI research include:

1. **Efficiency Over Scaling**: Researchers are prioritizing smaller, more efficient models that retain high performance with reduced computational costs. Institutions like DeepMind, Meta FAIR, and university labs have developed architectures achieving near-frontier performance at a tenth of the compute cost, addressing practical deployment challenges.

2. **Agentic AI in Production**: Agentic AI (systems that autonomously plan and execute tasks) has transitioned from demos to real-world workflows, impacting industries by enabling dynamic decision-making and automation. However, this shift introduces new risks, such as vendor lock-in through model routing across platforms.

3. **Frontier Models and Benchmarks**: OpenAI and Google’s large language models (LLMs) are now benchmarking closely, narrowing the gap in capabilities. This competition drives innovation but also raises concerns about infrastructure costs and security vulnerabilities in deployed systems.

4. **Responsible AI and Governance**: The Stanford HAI 2026 AI Index Report underscores efforts to balance AI’s transformative potential with ethical considerations, tracking progress in technical performance, economic impact, and societal challenges across sectors like healthcare, education, and policy.

5. **Security and Infrastructure Shifts**: The industry is grappling with security risks in AI systems and rethinking infrastructure to support scalable, secure, and sustainable deployments. Collaboration between developers and organizations is critical to mitigate risks and ensure equitable access to AI advancements.

For detailed insights, refer to the [Stanford AI Index Report](https://hai.stanford.edu/ai-index/2026-ai-index-report) and [Netguru’s 2026 AI Developments](https://www.netguru.com/blog/latest-ai-developments-2026).

In [90]:
stop_ollama("qwen3:8b")

Stopped Ollama model: qwen3:8b


In [101]:
query_2 = "what is the current AI industry trend, and what are the latest developments in AI research?"

response_2 = agent.invoke({"messages": [{"role": "user", "content": query_2}]})

In [102]:
print(response_2['messages'][-1].content)

As of 2026, the AI industry and research landscape is shaped by several key trends and developments, based on recent insights and reports:

---

### **Current AI Industry Trends (2026)**
1. **Ethical AI and Risk Management**  
   - Organizations are prioritizing AI safety, transparency, and fairness to mitigate risks like bias, job displacement, and security threats. EY’s "Four Futures of AI" framework highlights strategic debates around whether AI will empower or disrupt businesses and societies.

2. **Responsible AI Governance**  
   - Regulatory frameworks and corporate policies are evolving to ensure accountability, compliance, and ethical use of AI. This includes efforts to address environmental impacts (e.g., energy consumption of training models) and workforce upskilling.

3. **AI Integration into Scientific Research**  
   - Tools like LeapSpace (a platform combining AI with peer-reviewed research) are enabling scientists to analyze vast datasets and accelerate discoveries in f

In [103]:
display(Markdown(response_2['messages'][-1].content))

As of 2026, the AI industry and research landscape is shaped by several key trends and developments, based on recent insights and reports:

---

### **Current AI Industry Trends (2026)**
1. **Ethical AI and Risk Management**  
   - Organizations are prioritizing AI safety, transparency, and fairness to mitigate risks like bias, job displacement, and security threats. EY’s "Four Futures of AI" framework highlights strategic debates around whether AI will empower or disrupt businesses and societies.

2. **Responsible AI Governance**  
   - Regulatory frameworks and corporate policies are evolving to ensure accountability, compliance, and ethical use of AI. This includes efforts to address environmental impacts (e.g., energy consumption of training models) and workforce upskilling.

3. **AI Integration into Scientific Research**  
   - Tools like LeapSpace (a platform combining AI with peer-reviewed research) are enabling scientists to analyze vast datasets and accelerate discoveries in fields like medicine, climate science, and materials engineering. However, access to such tools remains a challenge due to paywalls or technical barriers.

4. **Economic and Societal Impact Analysis**  
   - The AI Index Report (Stanford HAI) emphasizes tracking AI’s role in economic growth, productivity, and societal changes. For example, AI-driven automation is reshaping industries, while concerns about inequality and job displacement persist.

5. **Global Collaboration and Competition**  
   - Nations and companies are investing heavily in AI R&D, with China and the U.S. leading in both technical innovation and infrastructure. However, there is growing emphasis on international cooperation to set global standards.

---

### **Latest AI Research Developments (2026)**
1. **Advances in Technical Performance**  
   - Breakthroughs in **foundation models** (e.g., larger, more efficient neural networks) and **multi-modal AI** (integrating text, images, and video) are enabling more versatile applications in healthcare, education, and creative industries.

2. **AI in Medicine and Science**  
   - AI is accelerating drug discovery, personalized medicine, and genomics research. For instance, AI-driven tools are now capable of predicting protein structures with near-atomic precision, speeding up biomedical innovations.

3. **AI for Sustainable Development**  
   - Research focuses on using AI to tackle climate change, such as optimizing renewable energy grids and monitoring deforestation. AI-powered climate models are improving weather forecasting and disaster response.

4. **AI Education and Accessibility**  
   - New tools are democratizing AI literacy, with platforms offering interactive courses and low-code AI development environments. This is critical for bridging the skills gap in emerging economies.

5. **AI Policy and Public Trust**  
   - Studies show mixed public sentiment: while many see AI as a tool for progress, concerns about privacy, surveillance, and algorithmic bias remain. Policymakers are working to balance innovation with safeguards.

---

### **Key Challenges**
- **Ethical and Regulatory Hurdles**: Balancing innovation with accountability remains a priority.
- **Access Inequality**: High costs of AI tools and data limit adoption in developing regions.
- **Security Risks**: AI systems are increasingly targeted by cyberattacks, necessitating robust defense mechanisms.

For the most up-to-date details, the **2026 AI Index Report** (Stanford HAI) and EY’s strategic analyses provide comprehensive data on these trends.

In [104]:
stop_ollama("qwen3:8b")

Stopped Ollama model: qwen3:8b


**V2 increased ContextWindow**

In [55]:
print(response_2['messages'][-1].content)

I currently do not have knowledge of events or developments after October 2024, so I cannot provide updates specific to 2026. However, based on trends up to October 2024, here’s a summary of AI industry trends and research developments:

### **AI Industry Trends (as of October 2024):**
1. **Enterprise Adoption**: Companies are increasingly integrating AI into workflows for automation, customer service, and data analysis. Tools like generative AI for content creation and code generation are becoming mainstream.
2. **AI Ethics and Regulation**: Governments and organizations are focusing on ethical AI practices, transparency, and regulations (e.g., EU’s AI Act) to address biases and ensure accountability.
3. **AI in Healthcare**: Breakthroughs in medical imaging, drug discovery, and personalized treatment plans using AI models are accelerating.
4. **Edge AI**: Deploying AI models on edge devices (e.g., smartphones, IoT) for real-time processing and reduced latency is gaining traction.
5. 

In [56]:
display(Markdown(response_2['messages'][-1].content))

I currently do not have knowledge of events or developments after October 2024, so I cannot provide updates specific to 2026. However, based on trends up to October 2024, here’s a summary of AI industry trends and research developments:

### **AI Industry Trends (as of October 2024):**
1. **Enterprise Adoption**: Companies are increasingly integrating AI into workflows for automation, customer service, and data analysis. Tools like generative AI for content creation and code generation are becoming mainstream.
2. **AI Ethics and Regulation**: Governments and organizations are focusing on ethical AI practices, transparency, and regulations (e.g., EU’s AI Act) to address biases and ensure accountability.
3. **AI in Healthcare**: Breakthroughs in medical imaging, drug discovery, and personalized treatment plans using AI models are accelerating.
4. **Edge AI**: Deploying AI models on edge devices (e.g., smartphones, IoT) for real-time processing and reduced latency is gaining traction.
5. **AI Collaboration**: Cross-industry partnerships (e.g., tech firms with healthcare providers) are driving innovation in specialized AI applications.

### **Latest AI Research (as of October 2024):**
- **Efficient Models**: Research into smaller, more efficient models (e.g., "model compression") to reduce computational costs while maintaining performance.
- **Multimodal AI**: Advances in models that process text, images, and video (e.g., CLIP, DALL-E) for better contextual understanding.
- **AI Safety**: Focus on aligning AI systems with human values and preventing harmful use cases (e.g., reinforcement learning from human feedback).
- **Quantum AI**: Early explorations of quantum computing’s potential to solve complex AI problems faster.

For updates beyond October 2024, I recommend checking recent industry reports or academic publications. Let me know if you’d like further details!

In [47]:
stop_ollama("qwen3:8b")

Stopped Ollama model: qwen3:8b
